# AutoPhOT example setup

This notebook runs AutoPhOT end to end on a folder of FITS images. Each section sets one group of parameters and explains what they do. Edit the paths and the target first; the rest can stay at the values shown until you have a reason to change them.

Every config key and its default live in `databases/default_input.yml`. `AutomatedPhotometry.load()` returns them as a nested dict that you override key by key.

## Imports and config

`AutomatedPhotometry.load()` reads the defaults. Everything below assigns into `autophot_input`, the dict the run consumes.

In [ ]:
import glob
import os
import sys

# Only needed when running from a source checkout instead of an installed package.
# sys.path.append('/path/to/autophot')

from autophot import AutomatedPhotometry

autophot_input = AutomatedPhotometry.load()

## Paths and target

`fits_dir` is scanned recursively for FITS images; flat and nested layouts both work (see Folder layout in the README). Outputs go to a `<fits_dir>_<outdir_name>` sibling. `wdir` is a working directory for caches, catalog queries, and the telescope database used in header lookup.

`target_name` resolves through TNS; set `target_ra` and `target_dec` to skip the lookup. `additional_targets` fits extra sources in the same frames. `selected_filters` limits the run to these bands, and `restart=False` skips images that already produced an `Output_*.csv`.

In [ ]:
autophot_input['fits_dir'] = '/path/to/photometry'
autophot_input['wdir'] = '/path/to/autophot_db'
autophot_input['outdir_name'] = 'REDUCED'

autophot_input['target_name'] = 'SN2026abc'
# Explicit coordinates instead of a TNS lookup:
# autophot_input['target_ra'] = 123.456789
# autophot_input['target_dec'] = -12.345678
# Extra sources fitted alongside the primary target:
# autophot_input['additional_targets'] = ['SN2021xyz']
# autophot_input['additional_targets_ra_dec'] = [[123.4, -12.3]]

autophot_input['selected_filters'] = ['g', 'r', 'i']
autophot_input['restart'] = False

## Calibration catalog

`use_catalog` picks the photometric reference. `'auto'` measures each image footprint once and keeps the catalog with the most usable calibrators per band; the choice is cached under `<wdir>/catalog_queries/`. A dict maps band groups to catalogs, and `'gaia_custom'` convolves Gaia XP spectra with your own throughput curves for bands no survey covers. `refcat` queries need MAST CasJobs credentials.

In [ ]:
autophot_input['catalog']['use_catalog'] = 'auto'
# Single catalog: 'gaia', 'panstarrs', 'sdss', 'apass', 'refcat', '2mass', ...
# Per-band mapping:
# autophot_input['catalog']['use_catalog'] = {
#     'grizJHK': 'refcat',
#     'u': 'gaia',
#     'UBVRI': 'apass',
#     'default': 'gaia',
# }

# refcat needs a MAST CasJobs workspace:
# autophot_input['catalog']['MASTcasjobs_wsid'] = '...'
# autophot_input['catalog']['MASTcasjobs_pwd'] = '...'

# Gaia XP + your own throughput curves (band -> local file or SVO filter ID):
# autophot_input['catalog']['transmission_curve_map'] = {'myband': '/path/to/myband.dat'}
# autophot_input['catalog']['curve_map_svo'] = {'r': 'SDSS/SDSS.r'}

## Preprocessing and photometry

`remove_cmrays` toggles cosmic-ray rejection, which is slow on large frames. `trim_image` crops an arcmin box around the target before the rest of the pipeline runs. `use_maximask` adds the CNN defect mask to the heuristic masks; it needs `maximask-and-maxitrack` plus tensorflow and runs far faster on a GPU.

`perform_emcee_fitting_s2n` is the S/N below which the MCMC fitter takes over from least-squares; a huge value runs MCMC on every detection. `psf_spatial_grid` builds a field-varying PSF on an NxN grid for wide or distorted fields.

In [ ]:
autophot_input['cosmic_rays']['remove_cmrays'] = 0
autophot_input['preprocessing']['trim_image'] = 5   # arcmin box around the target

_HAS_GPU = os.path.exists('/dev/kfd') or bool(glob.glob('/dev/nvidia*'))
autophot_input['background']['use_maximask'] = False
autophot_input['background']['maximask_allow_gpu'] = _HAS_GPU
autophot_input['background']['maximask_batch_size'] = 32 if _HAS_GPU else 4

autophot_input['photometry']['psf_oversample'] = 1
autophot_input['photometry']['perform_emcee_fitting_s2n'] = 100000
autophot_input['photometry']['apply_colorterms'] = False
# autophot_input['photometry']['psf_spatial_grid'] = 'auto'  # 'auto' or an int N

## WCS and distortion

`redo_wcs` re-solves each frame with astrometry.net, or with SCAMP when `solver` is `'scamp'`. `correct_distortion` adds a SCAMP + SWarp resampling pass for fields with real optical distortion; set `correct_distortion_resampling_type` to `'LANCZOS2'` on undersampled data.

In [ ]:
autophot_input['wcs']['redo_wcs'] = 1
# autophot_input['wcs']['solver'] = 'scamp'
autophot_input['wcs']['correct_distortion'] = False

## Limiting magnitudes

Artificial stars are injected and recovered at the configured S/N thresholds. `n_jobs` scales the bisection searches across processes and `inject_final_n_sites` sets how many farthest-point-spaced sites each iteration uses. `uncal_apply` rescales the reported photometric errors by the injection z-score scatter.

In [ ]:
autophot_input['limiting_magnitude']['n_jobs'] = 8
autophot_input['limiting_magnitude']['inject_final_n_sites'] = 4
# autophot_input['limiting_magnitude']['recovery_method'] = 'PSF'
# autophot_input['limiting_magnitude']['uncal_apply'] = True

## Template subtraction

`method` chooses `sfft`, `hotpants`, `zogy`, or `auto`. `forceconv='REF'` convolves the template to the science PSF so the science ePSF applies directly to the difference image. `kernel_order` sets the SFFT kernel polynomial degree: `0` is a constant kernel (the default) and `'auto'` adds spatial terms only when the matched-source evidence supports them.

`sfft_memory_policy` decides what happens when the predicted SFFT peak memory exceeds `sfft_max_memory_gb`: `'shrink'` reduces the kernel, `'fallback'` switches to HOTPANTS, `'ignore'` logs the estimate and runs the full kernel. `sfft_backend` selects `cupy` on GPU hosts or `numpy` otherwise, and `sfft_num_threads` caps BLAS threads per run. `match_max_sources` caps the matched-source list used for kernel fitting.

In [ ]:
autophot_input['template_subtraction']['do_subtraction'] = 1
autophot_input['template_subtraction']['alignment_method'] = 'spalipy'
autophot_input['template_subtraction']['method'] = 'sfft'   # 'sfft', 'hotpants', 'zogy', 'auto'

autophot_input['template_subtraction']['sfft_backend'] = 'cupy' if _HAS_GPU else 'numpy'
autophot_input['template_subtraction']['sfft_cuda_device'] = '0'
autophot_input['template_subtraction']['forceconv'] = 'REF'
autophot_input['template_subtraction']['kernel_order'] = '0'
autophot_input['template_subtraction']['zogy_psf'] = 'auto'

# 'shrink' reduces the kernel, 'fallback' uses HOTPANTS, 'ignore' runs it anyway:
autophot_input['template_subtraction']['sfft_memory_policy'] = 'fallback'
# autophot_input['template_subtraction']['sfft_max_memory_gb'] = 16
# autophot_input['template_subtraction']['sfft_num_threads'] = 4
# autophot_input['template_subtraction']['match_max_sources'] = 2000
# autophot_input['template_subtraction']['download_templates'] = 'panstarrs'

## Parallelism and verbosity

`nCPU` runs images in parallel subprocesses. `nGPU` pins those workers round-robin across the visible GPUs; `0` leaves the ambient device mask alone. `global_verbose_level` sets console output from `0` (quiet) to `3` (debug); the per-image `LOG_*.log` file records everything either way.

In [ ]:
autophot_input['nCPU'] = 1
autophot_input['nGPU'] = 0
autophot_input['global_verbose_level'] = 1   # 0 quiet / 1 normal / 2 verbose / 3 debug

## Run

Executes the pipeline and returns the path of the collected output CSV. Pass `do_photometry=False` to collect existing outputs without reprocessing.

In [ ]:
loc = AutomatedPhotometry.run_photometry(default_input=autophot_input)
print(loc)

## Lightcurve and tables

`plot_lightcurve` writes the detections-plus-limits figure and returns the detections CSV that `check_detection_plots` consumes. `plot_variability_check` compares the target's scatter against the reference-star ensemble to separate real variability from instrumental drift. `generate_photometry_table` writes an ASCII table.

In [ ]:
from lightcurve import (
    plot_lightcurve, plot_variability_check,
    check_detection_plots, generate_photometry_table,
)

_primary_name = autophot_input.get('target_name', 'Main target')

detections_loc = plot_lightcurve(
    loc, snr_limit=3, method='PSF', format='png', show=True,
    target_name=_primary_name,
)

try:
    plot_variability_check(loc, method='PSF', target_name=_primary_name, show=True)
except Exception as e:
    print(f'[WARNING] Variability check failed: {e}')

check_detection_plots(detections_loc, method='AP')
generate_photometry_table(loc, snr_limit=3, method='PSF', target_name=_primary_name)